# How Fast Do Markets React to Trade Announcements? 📉⚡

Welcome! In this notebook you'll replay the US-China trade war through the eyes of
the stock market. You'll pick real announcement dates — tariff threats, truces, a
famous Sunday tweet — and watch exactly how prices responded: how fast, how big,
and (most interestingly) *when they didn't react at all*.

**No coding or finance background needed.** Almost everything is pre-written. In
marked spots you'll edit a date, a number, or a list of stock tickers.

---

### ⚠️ Step zero — this is YOUR copy

This notebook is your personal copy — it has your name on the file. Your edits save
automatically. If it ever gets hopelessly tangled, **Ctrl+Z / Cmd+Z inside a cell
fixes almost everything** — and if it doesn't, email Heagen for a fresh copy.

### How this notebook works

- Cells run **top to bottom** — click a cell, press **Shift+Enter** (or ▶).
- **▶️ JUST RUN** cells — run, don't edit.
- **✏️ YOUR TURN** cells — change the line marked `# <-- EDIT`.
- Under every ✏️ cell is a **"What you should see"** box for checking yourself.

**Time:** built for ~4 hours. Setup + meet the data ~30 min · Beat 1 ~60 min ·
Beat 2 ~45 min · Beat 3 ~60 min · Stretch ~45 min.

## Setup

**▶️ JUST RUN** — loads tools and three and a half years of daily stock prices
(2017 through mid-2020, the heart of the trade war).

In [ ]:
# ▶️ JUST RUN — imports and data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/YOUR-ORG/trade-war-lab-notebooks/main/data/tariff_prices.csv"
prices = pd.read_csv(DATA_URL, index_col="date", parse_dates=True)

print(f"Loaded {len(prices)} trading days for {len(prices.columns)} tickers: {list(prices.columns)}")
prices.head()

> **What you should see:** `Loaded 880 trading days for 6 tickers: ['SPY', 'FXI',
> 'BA', 'CAT', 'SOYB', 'AAPL']` and a table of prices with dates down the side.

### Meet the cast

| Ticker | What it is | Why it's here |
|---|---|---|
| **SPY** | The S&P 500 — "the market" | Our baseline |
| **FXI** | Large Chinese companies | Direct China exposure |
| **BA** | Boeing | China was its biggest export customer |
| **CAT** | Caterpillar | Global industrial bellwether |
| **SOYB** | Soybeans | China's #1 retaliation target — aimed at US farmers |
| **AAPL** | Apple | Builds in China, sells in China |

And the six announcement dates we've pre-loaded. **▶️ JUST RUN:**

In [ ]:
# ▶️ JUST RUN — the six pre-loaded announcement dates
EVENTS = {
    "2018-03-22": "Tariff memo signed (escalation, surprise)",
    "2018-06-15": "25% tariffs on $50B announced (escalation, expected)",
    "2018-12-03": "Buenos Aires truce (de-escalation)",
    "2019-05-06": "Sunday tweet: tariffs to 25% (escalation, surprise)",
    "2019-08-23": "China retaliates; companies 'ordered' out (escalation, surprise)",
    "2020-01-15": "Phase One deal signed (de-escalation, fully expected)",
}
for d, label in EVENTS.items():
    print(f"{d}  —  {label}")

One more setup cell — the plotting helpers. **▶️ JUST RUN:**

In [ ]:
# ▶️ JUST RUN — helper tools
def _event_day(date):
    date = pd.Timestamp(str(date))
    if date not in prices.index:
        later = prices.index[prices.index >= date]
        if len(later) == 0:
            print(f"({date.date()} is past the end of this dataset.)"); return None
        print(f"({date.date()} wasn't a trading day — using {later[0].date()} instead.)")
        return later[0]
    return date

def event_plot(date, window, tickers=["SPY"]):
    """Prices around an announcement, rebased to 100 the day before it."""
    if isinstance(tickers, str):
        tickers = [tickers]
    window = int(window)
    day0 = _event_day(date)
    if day0 is None: return
    i = prices.index.get_loc(day0)
    if i == 0:
        print(f"({date} is the first day in this dataset, so there's no day before it "
              f"to compare against — try a later date.)")
        return
    chunk = prices.iloc[max(i - window, 0): i + window + 1]
    fig, ax = plt.subplots(figsize=(10, 5))
    for tk in tickers:
        if tk not in prices.columns:
            print(f"('{tk}' isn't in this dataset — options: {list(prices.columns)})"); continue
        rebased = chunk[tk] / prices[tk].iloc[i - 1] * 100
        ax.plot(range(-min(window, i), len(chunk) - min(window, i)), rebased, marker="o",
                markersize=3, label=tk)
        day0_ret = (prices[tk].iloc[i] / prices[tk].iloc[i - 1] - 1) * 100
        print(f"{tk}: day-0 move {day0_ret:+.1f}%")
    ax.axvline(0, color="red", linestyle="--", alpha=0.6, label="announcement day")
    label = EVENTS.get(str(date), str(date))
    ax.set_title(label); ax.set_xlabel("trading days from announcement")
    ax.set_ylabel("price (100 = day before)")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

def compare_events(date_a, date_b, ticker="SPY", window=5):
    """Overlay one ticker's path around two different announcements."""
    if ticker not in prices.columns:
        print(f"('{ticker}' isn't in this dataset — options: {list(prices.columns)})"); return
    window = int(window)
    fig, ax = plt.subplots(figsize=(10, 5))
    for date in [date_a, date_b]:
        day0 = _event_day(date)
        if day0 is None: return
        i = prices.index.get_loc(day0)
        if i == 0:
            print(f"({date} is the first day in this dataset, so there's no day before it "
                  f"to compare against — try a later date.)")
            return
        chunk = prices[ticker].iloc[max(i - window, 0): i + window + 1]
        rebased = chunk / prices[ticker].iloc[i - 1] * 100
        ax.plot(range(-min(window, i), len(chunk) - min(window, i)), rebased, marker="o", markersize=4,
                label=f"{date}: {EVENTS.get(str(date), '')[:38]}")
        day0_ret = (prices[ticker].iloc[i] / prices[ticker].iloc[i - 1] - 1) * 100
        print(f"{date} — {ticker} day-0 move: {day0_ret:+.1f}%")
    ax.axvline(0, color="red", linestyle="--", alpha=0.6)
    ax.set_title(f"{ticker} around two announcements")
    ax.set_xlabel("trading days from announcement"); ax.set_ylabel("price (100 = day before)")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

def scoreboard(tickers=["SPY", "FXI", "BA", "SOYB"]):
    """Day-0 percent move for every pre-loaded event, per ticker."""
    rows = []
    for d, label in EVENTS.items():
        day0 = _event_day(d)
        if day0 is None: continue
        i = prices.index.get_loc(day0)
        row = {"date": d, "event": label[:45]}
        for tk in tickers:
            row[tk] = f"{(prices[tk].iloc[i] / prices[tk].iloc[i-1] - 1) * 100:+.1f}%"
        rows.append(row)
    return pd.DataFrame(rows)

def market_adjusted(ticker, date, window=5):
    """Stretch helper: a stock's move MINUS the market's move (its 'abnormal' return)."""
    if ticker not in prices.columns:
        print(f"('{ticker}' isn't in this dataset — options: {list(prices.columns)})"); return
    window = int(window)
    day0 = _event_day(date)
    if day0 is None: return
    i = prices.index.get_loc(day0)
    if i == 0:
        print(f"({date} is the first day in this dataset, so there's no day before it "
              f"to compare against — try a later date.)")
        return
    stock = prices[ticker].iloc[i - 1: i + window + 1].pct_change().dropna()
    mkt = prices["SPY"].iloc[i - 1: i + window + 1].pct_change().dropna()
    abnormal = ((stock - mkt) * 100).cumsum()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(range(len(abnormal)), abnormal.values, marker="o")
    ax.axhline(0, color="gray", linestyle="--")
    ax.set_title(f"{ticker}'s move BEYOND the market, after {date}")
    ax.set_xlabel("trading days from announcement"); ax.set_ylabel("cumulative abnormal return (%)")
    ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    print(f"After {window} days, {ticker} moved {abnormal.iloc[-1]:+.1f}% beyond what the market explains.")

print("Helpers loaded ✅ — on to Beat 1!")

---
## Beat 1 — How fast? ✏️

August 23, 2019: China announces retaliation tariffs, and the president tweets that
American companies are "hereby ordered" to leave China. Let's watch the market find
out.

**✏️ YOUR TURN:** run as-is first, then change the date (copy any from the EVENTS
list) and the window.

In [ ]:
announcement_date = "2019-08-23"   # <-- EDIT (copy a date from the EVENTS list above)
window_in_days    = 10             # <-- EDIT (trading days shown on each side)

event_plot(announcement_date, window_in_days)

> **What you should see** (with the starting values): a price line that's wobbling
> along near 100, then **gaps down right at the red dashed line**, with a printed
> `SPY: day-0 move -2.7%`.
> **If you get a red error:** the date needs quotes and dashes — `"2019-08-23"` —
> and the window is a plain number, no quotes.

### Things to try

1. **Widen the window to 40.** Notice the drop doesn't get *spread out* — it stays a
   cliff at day 0. Markets don't slowly digest news over weeks; the reaction is
   concentrated in the first day (and much of it happened within *minutes* of the
   announcement — our daily data is actually too coarse to see how fast it really was).
2. Try `"2018-03-22"` — the very first big tariff announcement. Compare the size of
   its cliff to August 2019's.
3. Look at the days *before* the red line. Any hint of the drop coming? If markets
   moved before announcements, what would that suggest? (Keep this thought — it
   returns with force in Beat 3.)

---
## Beat 2 — Are up and down symmetric? ✏️

The trade war had escalations *and* de-escalations. If markets treated good and bad
news equally, a truce should bounce prices up about as much as a threat knocks them
down. Does it?

**✏️ YOUR TURN:** compare an escalation to a de-escalation.

In [ ]:
first_event  = "2019-08-23"   # <-- EDIT (an escalation)
second_event = "2018-12-03"   # <-- EDIT (the Buenos Aires truce)

compare_events(first_event, second_event)

> **What you should see** (with the starting dates): two overlaid lines — one
> dropping hard at day 0, one rising modestly — with printed day-0 moves of roughly
> `-2.7%` versus `+1.2%`.
> **If you get a red error:** both dates need quotes and dashes.

### Things to try

1. Swap in the other escalations. Is the pattern general — do drops on bad news run
   bigger than pops on good news?
2. Why might that be? Two candidate stories: (a) *bad news is treated as more
   informative* — a threat proves relations are worse than believed, while a truce
   might not hold; (b) *loss aversion* — investors fear losses more than they enjoy
   gains, so selling panics run harder than buying rallies. The data can't pick the
   winner for you, but it's real: asymmetric reaction to news is one of the
   best-documented patterns in finance.
3. Run the same comparison with `ticker="SOYB"` added inside the parentheses:
   `compare_events(first_event, second_event, ticker="SOYB")`. Soybeans were China's
   chosen retaliation weapon — aimed at US farmers. Does the asymmetry look different
   for the sector in the crosshairs?

---
## Beat 3 — Surprise vs. expected ✏️

Two de-escalation... wait, no — two events, one lesson. May 5, 2019: a **Sunday
tweet** out of nowhere re-escalates the trade war. January 15, 2020: the **Phase One
deal** — negotiated publicly for months, signing date known well in advance — is
finally signed.

One is a surprise. One is the most anticipated event of the whole trade war. Watch
what that difference does.

**✏️ YOUR TURN:**

In [ ]:
surprise_event = "2019-05-06"   # <-- EDIT
expected_event = "2020-01-15"   # <-- EDIT
tickers        = ["SPY", "FXI", "BA"]   # <-- EDIT this list

event_plot(surprise_event, 5, tickers)
event_plot(expected_event, 5, tickers)

> **What you should see** (with the starting values): the first chart cliffs at the
> red line (day-0 moves around `-1.6%` to `-3%` depending on ticker). The second
> chart — the *biggest deal of the entire trade war* — barely twitches: day-0 moves
> around `-0.3%` to `+0.1%`.
> **If you get a red error:** check the list shape — `["SPY", "FXI"]`, each ticker
> in its own quotes.

### Things to try

1. Sit with the weirdness: signing the deal that paused the trade war moved markets
   *less* than one tweet. The resolution: by January 15, everyone already knew the
   deal was coming — so everyone who wanted to buy on the news **already had**,
   weeks earlier. The information entered prices when it became *known*, not when it
   became *official*. Traders call this "priced in."
2. So what moves markets isn't news — it's **surprise**, the gap between what
   happened and what was expected. Re-read the EVENTS list with that lens: rank the
   six by how surprising they were, then check your ranking with the scoreboard below.
3. Swap the tickers list to `["SOYB", "CAT", "AAPL"]` and re-run both. Which companies
   had the most riding on surprise?

**▶️ JUST RUN — the full scoreboard:**

In [ ]:
# ▶️ JUST RUN — day-0 moves for all six events
scoreboard()

> **What you should see:** a 6-row table, one event per row, with signed percentage
> moves per ticker. The surprise escalations dominate; Phase One's row is nearly flat.

---
## Stretch goal (optional) — beyond the market 🔬

When Boeing drops 4% on an announcement day, how much of that is *Boeing* and how
much is just... everything dropping? Professionals answer by subtracting the
market's move — what's left is the stock's **abnormal return**. You just learned the
core tool of real event-study research.

**✏️ YOUR TURN:**

In [ ]:
ticker = "BA"            # <-- EDIT
event  = "2019-08-23"    # <-- EDIT

market_adjusted(ticker, event)

> **What you should see:** a line starting at 0 and drifting away from it, plus a
> printed line like `After 5 days, BA moved -2.1% beyond what the market explains.`

Try every ticker on the same event. Which company's fate was most tied to that
specific announcement, once you strip out the marketwide panic? Does the answer
match what you'd guess from the "meet the cast" table?

---
## What you just did

With a few edited dates and lists, you:

- watched news hit prices as a cliff, not a slope — reaction concentrated at day 0,
- found the asymmetry: markets punish escalation harder than they reward peace,
- discovered "priced in" — the biggest deal of the trade war moved markets less than
  one Sunday tweet, because *surprise*, not news, is what moves prices,
- and computed abnormal returns, the actual workhorse of event-study research.

**The bridge to Week 2:** every date you plotted traces back to *documents* —
memos, proclamations, bills moving through Congress. Markets react in minutes, but
someone has to read the paper trail to know what's coming. The lab's classifier
exists to read that legislative paper trail at scale — and next week, you label the
bills that make it work.

**Before you close:** write down (1) the single most surprising chart you made, and
(2) your ranking of the six events by surprise. Bring both to the Week 2 kickoff.